In [ ]:
truths = []

for example in test_data[:1]:
    events = {}
    times = {}
    event_quins = {}
    ee_trips = []
    ets = {}
    for instance in example['instances']:
        instance_id = instance["id"]
        if instance["type"] == "EVENT":
            events[instance_id] = instance
        else:
            times[instance_id] = instance

    for ee in example["ee_temprels"]:
        ee_trips.append({"event1_id":ee["e1"], "temp_relation":ee["rel"], "event2_id":ee["e2"]})

    for et in example["event_times"]:
        evid = et["event"] 
        if 'value' in times[et["time"]]:
            value = times[et["time"]]['value']
        else:
            value = None
        if evid not in ets:
            ets[evid] = [value]
        else:
            ets[evid].append(value)

    for eid, event in events.items():
        quint = {
            "id": eid,
            "event": event["text"],
            "subject": None,
            "object": None,
            "times": ets.get(eid, [])
        }
        event_quins[eid] = quint

    truths.append({'times':list(times.values()), 'quintuples':list(event_quins.values()), 'triples':list(ee_trips)})


In [ ]:
import isodate

def gentext_to_iso8601(gentext: str):
    parsers = {
        isodate.parse_date,
        isodate.parse_datetime,
        isodate.parse_time,
        isodate.parse_duration,
    }
    for parser in parsers:
        try:
            output = parser(gentext)
            if output is not None:
                return output
        except Exception:
            continue
        return None
    
def get_start_end_times(event_times: list):
    for time in event_times:
        isoobj = gentext_to_iso8601(time)
        print(isoobj)

In [ ]:
time_scores =[]
from copy import deepcopy
def time_match(all_ground, all_pred):
    ap = deepcopy(all_pred)
    prediction_scores = []
    for t in all_ground:
        if t['id'] == 0:
            continue
        value_score = "NONE"
        match_score = "NONE"
        type_score = "NONE"
        for p in ap:
            if t['text'] in [ptext[1] for ptext in ap]:
                ap.remove(p)
                match_score = "MATCH"

                if t['value'] == p[2]:
                    value_score = "CORRECT"
                elif t['value'] is not None and p[2] is None:
                    value_score = "HALLUCINATED"
                
                if p[3] in ['DATE', 'DURATION', 'SET', 'TIME', 'GEO_TIME']:
                    type_score = p[3]
                else:
                    type_score = "HALLUCINATED"
                prediction_scores.append((match_score, value_score, type_score, t['type']))
                break
        prediction_scores.append((match_score, value_score, type_score, "NONE"))
    return prediction_scores

for truth, pred in zip(truths, out.values()):
    time_scores.extend(time_match(truth['times'], pred['pred']['times']))

time_scores

In [ ]:
from sklearn.metrics import f1_score

corrs = [1 if ts[0]=="MATCH" else 0 for ts in time_scores]
f1 = f1_score([1]*len(corrs), corrs, average='binary')
f1